# Day 09 Tutorial — Advanced SQL Windows & Analytics


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
spark.createDataFrame(
    [
        ('c1', '2024-01-01', 100),
        ('c1', '2024-01-02', 150),
        ('c1', '2024-01-03', 120),
        ('c2', '2024-01-01', 80),
        ('c2', '2024-01-02', 90),
    ],
    ['customer_id', 'dt', 'amount'],
).createOrReplaceTempView('daily_sales')


In [ ]:
spark.sql('''
SELECT customer_id, dt, amount,
       LAG(amount) OVER (PARTITION BY customer_id ORDER BY dt) AS prev_amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY dt
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS amount_rn
FROM daily_sales
ORDER BY customer_id, dt
''').show()


In [ ]:
spark.sql('''
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY dt DESC) AS rn
  FROM daily_sales
)
SELECT customer_id, dt, amount FROM ranked WHERE rn = 1
''').show()
